In [1]:
# conda activate anndata

import os
import sys
import anndata as ad

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

In [ ]:
adata = ad.read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_SJ_counts.h5ad")

In [ ]:
expr = pd.read_csv("data/cleaned/TopModPosFDR/GTEx_cortex_counts_TMMF_All_501_outliers_removed_42147genes_cleaned.csv", index_col=0)

In [ ]:
expr.columns = expr.columns.str.replace(".","-")

In [3]:
adata.shape

(772, 233295)

In [ ]:
adata.var_names

In [4]:
SJ_counts_table = pd.DataFrame(adata.X.T, columns=adata.obs_names, index=adata.var_names)

In [6]:
SJ_counts_table.shape

(233295, 772)

In [ ]:
# Subset to just the samples we want to use for PSI calculation

In [13]:
SJ_counts_table = SJ_counts_table[expr.columns]

In [14]:
SJ_counts_table.shape

(233295, 501)

In [34]:
minSamples = 2
minJR = 1

In [ ]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events

# I1_filt = I1_table.index[(I1_table > minJR).astype(int).sum(axis=1) > minSamples]
# I2_filt = I2_table.index[(I2_table > minJR).astype(int).sum(axis=1) > minSamples]
# SE_filt = SE_table.index[(SE_table > minJR).astype(int).sum(axis=1) > minSamples]
    
I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]
filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [36]:
psi.shape[0]

31721

In [37]:
psi.head()

,GTEX-111FC-0011-R3b-SM-GJ3PN,GTEX-117XS-0011-R3a-SM-GIN8W,GTEX-1192X-0011-R3b-SM-GIN8Y,GTEX-11DXW-0011-R3b-SM-DNZZE,GTEX-11GS4-0011-R3b-SM-GJ3RI,GTEX-11GSO-0011-R3b-SM-57WB2,GTEX-11GSP-0011-R3a-SM-9QEGF,GTEX-11NUK-0011-R3b-SM-GJ3RO,GTEX-11NV4-0011-R3b-SM-GINAJ,GTEX-11O72-0011-R3a-SM-H65ZL,...,GTEX-XLM4-0011-R10A-SM-4AT5P,GTEX-Y8DK-0011-R10A-SM-4SOK1,GTEX-YJ89-0011-R10a-SM-4SOK9,GTEX-ZF28-0011-R10a-SM-4WWEH,GTEX-ZUA1-0011-R10a-SM-51MT6,GTEX-ZV68-0011-R10a-SM-51MT7,GTEX-ZVT3-0011-R10b-SM-57WB6,GTEX-ZVZQ-0011-R10b-SM-51MRT,GTEX-ZXG5-0011-R10a-SM-57WDD,GTEX-ZZPT-0011-R10b-SM-GPI8B
ENSG00000292994_other_1,1.000000,1.0,1.0,1.0,1.000000,1.000000,0.000000,0.500000,1.0,0.0,...,0.666667,1.0,1.0,0.875000,1.0,0.5,1.000000,1.000000,1.0,1.0
ENSG00000290385_other_1,0.111111,0.5,0.5,0.0,0.333333,0.000000,0.000000,0.090909,0.0,0.0,...,0.000000,0.0,0.0,0.142857,0.0,0.0,0.090909,0.142857,0.0,0.0
ENSG00000290385_other_2,0.272727,0.5,0.5,0.0,0.600000,0.000000,0.076923,0.090909,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,1.0,0.230769,0.142857,0.0,0.0
ENSG00000290385_other_3,1.000000,1.0,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.0,...,1.000000,1.0,1.0,1.000000,0.5,0.0,1.000000,1.000000,1.0,1.0
ENSG00000290385_other_4,1.000000,0.0,1.0,1.0,1.000000,0.666667,1.000000,1.000000,1.0,1.0,...,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000,1.000000,1.0,0.0


In [10]:
psi.to_csv(f"data/GTEx_cortex_exon_PSI.csv")
reads.to_csv(f"data/GTEx_cortex_exon_counts.csv")